# Mission 04 — Part 1: Systems Architect
## DPO Loss Core & Log-Likelihood Extraction (USSE Noosphere, `SEC-04-ORI`)

Deliverables: vectorized `get_batch_logps`, policy-vs-reference log-ratios, `DPOLoss` in pure PyTorch (`F.logsigmoid`, no TRL), and an empirical gradient test with a real trainable policy and a real frozen reference model.

$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}\left[\log \sigma\big(\beta(\Delta_w - \Delta_l)\big)\right],\quad \Delta_y = \log\pi_\theta(y\mid x) - \log\pi_{\text{ref}}(y\mid x)$$

The implementation lives in `01_DPO_Loss_Core.py`; this notebook imports it and runs each check.

In [1]:
import importlib.util, copy, math, pathlib
import torch, torch.nn.functional as F

spec = importlib.util.spec_from_file_location("dpo_core", pathlib.Path("01_DPO_Loss_Core.py"))
core = importlib.util.module_from_spec(spec); spec.loader.exec_module(core)
get_batch_logps, DPOLoss, dpo_log_ratios = core.get_batch_logps, core.DPOLoss, core.dpo_log_ratios
print("torch", torch.__version__)

torch 2.15.0+cu132


### 1. `get_batch_logps` — `torch.gather` + padding mask
Signature is unchanged: `get_batch_logps(logits, labels, label_pad_token_id=-100, average_log_prob=False)`. Labels are shifted **inside** (logits `[:, :-1]` vs labels `[:, 1:]`); prompt and padding tokens carry `-100`.
We compare it against a slow, explicit Python-loop implementation.

In [2]:
torch.manual_seed(42)
B, T, V = 4, 16, 128
logits = torch.randn(B, T, V)
labels = torch.randint(0, V, (B, T))
labels[:, :4] = -100      # prompt tokens masked
labels[1, 10:] = -100     # right padding on row 1
labels[3, :] = -100       # fully masked row

fast = get_batch_logps(logits, labels)
slow = core.naive_sequence_logp(logits, labels)
print("vectorized :", [round(x, 4) for x in fast.tolist()])
print("naive loop :", [round(x, 4) for x in slow.tolist()])
print("max |diff| :", (fast.double() - slow).abs().max().item())

# logits that only predict masked targets must not matter
logits2 = logits.clone(); logits2[1, 9:, :] = torch.randn(T - 9, V) * 100
print("padding-invariant:", torch.allclose(get_batch_logps(logits2, labels), fast))

avg = get_batch_logps(logits, labels, average_log_prob=True)
n_tok = (labels[:, 1:] != -100).sum(-1).clamp(min=1)
print("avg == sum / n_tokens:", torch.allclose(avg, fast / n_tok), "| masked row ->", fast[3].item())

vectorized : [-69.5087, -33.7169, -65.6692, 0.0]
naive loop : [-69.5087, -33.7169, -65.6692, 0.0]
max |diff| : 1.4795930241007227e-06
padding-invariant: True
avg == sum / n_tokens: True | masked row -> 0.0


### 2. Policy-vs-reference log-ratios
`dpo_log_ratios` returns $\Delta_w$ and $\Delta_l$; `DPOLoss` uses it internally.

In [3]:
pc, pr = torch.tensor([-10., -12., -9.]), torch.tensor([-11., -10., -9.5])
rc, rr = torch.tensor([-11., -12., -10.]), torch.tensor([-10., -10., -9.])
dw, dl = dpo_log_ratios(pc, pr, rc, rr)
print("Delta_w:", dw.tolist()); print("Delta_l:", dl.tolist())
print("beta*(Dw-Dl):", (0.1 * (dw - dl)).tolist())

Delta_w: [1.0, 0.0, 1.0]
Delta_l: [-1.0, 0.0, -0.5]
beta*(Dw-Dl): [0.20000000298023224, 0.0, 0.15000000596046448]


### 3. `DPOLoss` — numerical stability and analytic checks
A naive $-\log\sigma(z)$ overflows to `inf` for large negative $z$; `F.logsigmoid` stays finite.

In [4]:
z = torch.tensor([-1e4, -100., 0., 100., 1e4], requires_grad=True)
print("naive  -log(sigmoid):", (-torch.log(torch.sigmoid(z))).tolist())
print("stable -logsigmoid  :", (-F.logsigmoid(z)).tolist())
(-F.logsigmoid(z)).sum().backward(); print("stable grads finite  :", bool(torch.isfinite(z.grad).all()))

# analytic gradient: dL/dlogp_w = -beta * sigmoid(-z) / B
torch.manual_seed(0); beta = 0.1
pc = (torch.randn(6) * 5).requires_grad_(); pr = (torch.randn(6) * 5).requires_grad_()
rc, rr = torch.randn(6) * 5, torch.randn(6) * 5
loss, m = DPOLoss(beta)(pc, pr, rc, rr); loss.backward()
sig = torch.sigmoid(-beta * ((pc - rc) - (pr - rr))).detach()
print("grad matches analytic:", torch.allclose(pc.grad, -beta * sig / 6, atol=1e-6), torch.allclose(pr.grad, beta * sig / 6, atol=1e-6))
eq = DPOLoss(beta)(*[torch.zeros(3)] * 4)[0]
print(f"policy == reference -> loss {eq.item():.4f} (ln 2 = {math.log(2):.4f})")
print({k: round(v.item(), 4) for k, v in m.items()})

naive  -log(sigmoid): [inf, inf, 0.6931471824645996, -0.0, -0.0]
stable -logsigmoid  : [10000.0, 100.0, 0.6931471824645996, 3.923635700109488e-44, -0.0]
stable grads finite  : True
grad matches analytic: True True
policy == reference -> loss 0.6931 (ln 2 = 0.6931)
{'loss': 0.8237, 'chosen_win_rate': 0.5, 'reward_margin': -0.0427, 'bt_probability': 0.5012, 'chosen_rewards': -0.1623, 'rejected_rewards': -0.1197}


### 4. Gradient test with a real policy and a real frozen reference
`TinyCausalLM` is a 2-layer decoder-only transformer in pure PyTorch (no downloads). The reference is frozen (`requires_grad=False`, `eval()`, forward under `torch.no_grad()`); the optimizer receives **only policy parameters**. We snapshot both models and check after the optimizer steps.

In [5]:
torch.manual_seed(42)
reference = core.TinyCausalLM(vocab_size=64)
policy = copy.deepcopy(reference)                 # pi_theta initialised at pi_ref
for p in reference.parameters(): p.requires_grad_(False)
reference.eval()
ref_snap = {k: v.clone() for k, v in reference.state_dict().items()}
pol_snap = {k: v.clone() for k, v in policy.state_dict().items()}

batch = core.make_preference_batch(vocab_size=64)
criterion = DPOLoss(beta=0.1)
opt = torch.optim.AdamW(policy.parameters(), lr=1e-3)

def step():
    opt.zero_grad(set_to_none=True)
    pc = core.compute_sequence_logps(policy, *batch["chosen"])
    pr = core.compute_sequence_logps(policy, *batch["rejected"])
    with torch.no_grad():
        rc = core.compute_sequence_logps(reference, *batch["chosen"])
        rr = core.compute_sequence_logps(reference, *batch["rejected"])
    loss, m = criterion(pc, pr, rc, rr); loss.backward(); opt.step()
    return loss.item(), m

l0, m0 = step()
print(f"step 0: loss={l0:.4f} (ln2={math.log(2):.4f}), margin={m0['reward_margin'].item():+.5f}")
print("policy tensors with grad   :", sum(p.grad is not None for p in policy.parameters()), "/", len(list(policy.parameters())))
print("reference tensors with grad:", sum(p.grad is not None for p in reference.parameters()), "/", len(list(reference.parameters())))

step 0: loss=0.6931 (ln2=0.6931), margin=-0.00000
policy tensors with grad   : 29 / 29
reference tensors with grad: 0 / 29


In [6]:
history = [l0]
for i in range(1, 30):
    l, m = step(); history.append(l)
    if i % 5 == 0: print(f"step {i:2d}  loss={l:.4f}  win-rate={m['chosen_win_rate'].item():.2f}  margin={m['reward_margin'].item():+.4f}")

ref_same = all(torch.equal(ref_snap[k], v) for k, v in reference.state_dict().items())
changed = sum(not torch.equal(pol_snap[k], v) for k, v in policy.state_dict().items())
print("\nreference bit-identical after 30 optimizer steps:", ref_same)
print(f"policy tensors changed: {changed}/{len(pol_snap)}")
print(f"loss {history[0]:.4f} -> {history[-1]:.4f}")
assert ref_same and changed > 0 and history[-1] < history[0]

step  5  loss=0.4991  win-rate=1.00  margin=+0.4375
step 10  loss=0.3636  win-rate=1.00  margin=+0.8353


step 15  loss=0.2711  win-rate=1.00  margin=+1.1920


step 20  loss=0.2055  win-rate=1.00  margin=+1.5213
step 25  loss=0.1578  win-rate=1.00  margin=+1.8296



reference bit-identical after 30 optimizer steps: True
policy tensors changed: 29/29
loss 0.6931 -> 0.1288


### 5. Full script self-test
`python 01_DPO_Loss_Core.py` runs all of the above as one verification harness.

In [7]:
core.run_verification()

USSE NOOSPHERE • SUBSYSTEM 04 CALIBRATION: DPO LOSS CORE TEST

[1] get_batch_logps (torch.gather + padding mask)
    matches naive loop (max |diff| = 1.48e-06)
    fully-masked row -> 0.0 (no NaN)
    logits feeding masked targets do not change the result
    average_log_prob=True equals sum / n_response_tokens

[2] Numerical stability: -logsigmoid vs naive -log(sigmoid)
    z       : [-10000.0, -100.0, 0.0, 100.0, 10000.0]
    naive   : [inf, inf, 0.6931471824645996, -0.0, -0.0]
    stable  : [10000.0, 100.0, 0.6931471824645996, 3.923635700109488e-44, -0.0]
    stable loss and gradients finite at |z| = 1e4 (naive version gives inf)

[3] DPOLoss analytic checks
    loss == -mean(log sigma(beta(Dw - Dl))); d loss/d logp_w = -beta*sigma(-z)/B (analytic)
    policy == reference -> loss = ln 2 = 0.6931

[4] Gradient test with real models: trainable policy vs frozen reference
    step 0: loss = 0.6931 (= ln 2), reward margin = -0.0000 (policy == reference)
    policy: 29/29 parameter tensor

    after 30 AdamW steps: reference weights bit-identical to snapshot (frozen)
    after 30 AdamW steps: 29/29 policy tensors changed
    loss 0.6931 -> 0.1351; chosen win-rate 1.00, reward margin +1.9936

[5] Sanity: reference stays frozen even if it is (wrongly) put in the optimizer
    optimizer.step() on grad-less reference parameters is a no-op

[SUCCESS] Subsystem 04 DPO Core module successfully validated.


### Notes
* Reference log-probs are computed under `torch.no_grad()`, and the reference has `requires_grad=False`; both guard against accidental gradient flow.
* `DPOLoss.forward` keeps its original signature and metric keys (`loss`, `chosen_win_rate`, `reward_margin`, `bt_probability`, `chosen_rewards`, `rejected_rewards`), so Parts 2 and 3 can reuse it.
* Test models are tiny and randomly initialised, so the numbers demonstrate correctness of the mechanics, not alignment quality.